# Tushare 数据下载：指数/ETF 日频行情

本 notebook 是当前择时策略项目专用的数据下载脚本。

项目只需要单一指数或 ETF 的日频 OHLCV 数据，保存为 `data/raw/*.parquet` 后即可传给后端或 CLI。

最小字段要求：`trade_date, open, high, low, close, vol, amount`。

注意：一个输出文件只保存一个 `ts_code`，不要把多个标的混在同一个 Parquet 里。


In [1]:
import time
from pathlib import Path

import pandas as pd
import tushare as ts


## 1. 项目路径与下载参数

如果你在 `Timing_Strategy/data` 目录打开 notebook，下面会自动识别项目根目录；如果你在项目根目录打开，也能正常工作。


In [2]:
# 下载区间
START_DATE = "20230101"
END_DATE = pd.Timestamp.today().strftime("%Y%m%d")

# 请求间隔，避免接口请求过快
SLEEP_SECONDS = 0.3

# 自动识别项目根目录
CURRENT_DIR = Path.cwd().resolve()
if CURRENT_DIR.name == "data":
    PROJECT_ROOT = CURRENT_DIR.parent
elif (CURRENT_DIR / "timing_strategy").exists():
    PROJECT_ROOT = CURRENT_DIR
else:
    PROJECT_ROOT = CURRENT_DIR.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("当前工作目录：", CURRENT_DIR)
print("项目根目录：", PROJECT_ROOT)
print("数据输出目录：", RAW_DIR)
print("START_DATE =", START_DATE)
print("END_DATE =", END_DATE)


当前工作目录： /Users/linaismith/Desktop/实习/论文复刻/Timing_Strategy/data
项目根目录： /Users/linaismith/Desktop/实习/论文复刻/Timing_Strategy
数据输出目录： /Users/linaismith/Desktop/实习/论文复刻/Timing_Strategy/data/raw
START_DATE = 20230101
END_DATE = 20260528


## 2. 初始化 Tushare API

API Token 已保存在下面的 `TOKEN` 变量中。不要把 notebook 或 token 发给无关人员。


In [3]:
TOKEN = "11b7ca46e76c8936bb8324686c8b715a45a7828bbc4ef693d61c38cda7a3"
TUSHARE_HTTP_URL = "http://jiaoch.site"

if not TOKEN:
    raise ValueError("请先在 TOKEN 中填写 Tushare API Token")

pro = ts.pro_api(TOKEN)

# 如果你使用的是代理/镜像接口，保留下面两行；如果使用官方接口，可以把 TUSHARE_HTTP_URL 设为空字符串。
pro._DataApi__token = TOKEN
if TUSHARE_HTTP_URL:
    pro._DataApi__http_url = TUSHARE_HTTP_URL

print("Tushare API 初始化完成")


Tushare API 初始化完成


## 3. 配置要下载的标的

`api` 可选：

- `index_daily`：指数日线，例如沪深300指数 `399300.SZ`
- `fund_daily`：ETF日线，例如沪深300ETF `510300.SH`

可以在 `TARGETS` 中保留一个标的，也可以配置多个标的；系统会分别保存为多个 Parquet 文件。


In [4]:
TARGETS = [
    {
        "name": "513860 ETF",
        "ts_code": "513860.SH",
        "api": "fund_daily",
        "output_name": "513860_etf.parquet",
    },
    # 如果之后还想同时下载沪深300指数，可以取消下面几行注释。
    # {
    #     "name": "沪深300指数",
    #     "ts_code": "399300.SZ",
    #     "api": "index_daily",
    #     "output_name": "hs300_index.parquet",
    # },
]

REQUIRED_FIELDS = ["trade_date", "open", "high", "low", "close", "vol", "amount"]
TUSHARE_FIELDS = "ts_code," + ",".join(REQUIRED_FIELDS)

TARGETS


[{'name': '513860 ETF',
  'ts_code': '513860.SH',
  'api': 'fund_daily',
  'output_name': '513860_etf.parquet'}]

## 4. 下载与标准化函数


In [5]:
def safe_call(func, max_retry=3, sleep_seconds=1, **kwargs):
    """接口请求失败时自动重试。"""
    last_error = None
    for i in range(max_retry):
        try:
            df = func(**kwargs)
            if df is None:
                return pd.DataFrame()
            return df
        except Exception as exc:
            last_error = exc
            print(f"第 {i + 1} 次请求失败：{exc}")
            time.sleep(sleep_seconds)
    raise RuntimeError(f"多次请求失败：{last_error}")


def get_api_func(api_name: str):
    if api_name == "index_daily":
        return pro.index_daily
    if api_name == "fund_daily":
        return pro.fund_daily
    raise ValueError(f"不支持的 Tushare 接口：{api_name}")


def normalize_for_project(df: pd.DataFrame, target: dict) -> pd.DataFrame:
    """保留项目需要的字段，并确保单文件只有一个 ts_code。"""
    if df.empty:
        raise ValueError(f"{target['name']} 下载结果为空，请检查 ts_code、日期区间或接口权限")

    missing = [field for field in REQUIRED_FIELDS if field not in df.columns]
    if missing:
        raise ValueError(f"{target['name']} 缺少字段：{missing}")

    df = df.copy()
    if "ts_code" in df.columns:
        unique_codes = df["ts_code"].dropna().astype(str).unique().tolist()
        if len(unique_codes) != 1:
            raise ValueError(f"{target['name']} 不是单一 ts_code 数据：{unique_codes}")

    df = df[["ts_code", *REQUIRED_FIELDS]].copy()
    df["trade_date"] = df["trade_date"].astype(str)
    df = df.sort_values("trade_date").drop_duplicates("trade_date").reset_index(drop=True)

    for col in ["open", "high", "low", "close", "vol", "amount"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    if df[REQUIRED_FIELDS].isna().any().any():
        raise ValueError(f"{target['name']} 存在缺失值或无法转换为数字的字段")

    return df


## 5. 执行下载并保存 Parquet


In [6]:
downloaded = []

for target in TARGETS:
    api_func = get_api_func(target["api"])
    save_path = RAW_DIR / target["output_name"]

    print(f"开始下载：{target['name']} {target['ts_code']} -> {save_path}")

    raw_df = safe_call(
        api_func,
        ts_code=target["ts_code"],
        start_date=START_DATE,
        end_date=END_DATE,
        fields=TUSHARE_FIELDS,
    )

    df = normalize_for_project(raw_df, target)
    df.to_parquet(save_path, index=False)

    downloaded.append({
        "name": target["name"],
        "ts_code": target["ts_code"],
        "api": target["api"],
        "rows": len(df),
        "start": df["trade_date"].min(),
        "end": df["trade_date"].max(),
        "path": str(save_path),
    })

    print(f"保存完成：{save_path}，行数：{len(df)}")
    time.sleep(SLEEP_SECONDS)

summary_df = pd.DataFrame(downloaded)
summary_df


开始下载：513860 ETF 513860.SH -> /Users/linaismith/Desktop/实习/论文复刻/Timing_Strategy/data/raw/513860_etf.parquet
保存完成：/Users/linaismith/Desktop/实习/论文复刻/Timing_Strategy/data/raw/513860_etf.parquet，行数：820


,name,ts_code,api,rows,start,end,path
0,513860 ETF,513860.SH,fund_daily,820,20230103,20260527,/Users/linaismith/Desktop/实习/论文复刻/Timing_Strat...


## 6. 检查保存结果

下面会读取刚保存的 Parquet，确认字段和项目读取器兼容。


In [7]:
for item in downloaded:
    path = Path(item["path"])
    df = pd.read_parquet(path)
    print("文件：", path)
    print("字段：", df.columns.tolist())
    print("行数：", len(df), "日期：", df["trade_date"].min(), "->", df["trade_date"].max())
    display(df.head())
    display(df.tail())


文件： /Users/linaismith/Desktop/实习/论文复刻/Timing_Strategy/data/raw/513860_etf.parquet
字段： ['ts_code', 'trade_date', 'open', 'high', 'low', 'close', 'vol', 'amount']
行数： 820 日期： 20230103 -> 20260527


,ts_code,trade_date,open,high,low,close,vol,amount
0,513860.SH,20230103,0.515,0.528,0.505,0.527,707903.0,36755.736
1,513860.SH,20230104,0.531,0.539,0.526,0.538,905209.0,48363.837
2,513860.SH,20230105,0.551,0.560,0.549,0.550,1070270.0,59162.322
3,513860.SH,20230106,0.557,0.557,0.537,0.538,722812.0,39547.200
4,513860.SH,20230109,0.543,0.550,0.540,0.545,573659.0,31265.307


,ts_code,trade_date,open,high,low,close,vol,amount
815,513860.SH,20260521,0.633,0.640,0.619,0.620,2667374.20,168020.424
816,513860.SH,20260522,0.627,0.633,0.623,0.631,3098548.00,194727.016
817,513860.SH,20260525,0.628,0.630,0.627,0.628,488354.01,30690.663
818,513860.SH,20260526,0.636,0.642,0.627,0.641,5426230.00,344617.286
819,513860.SH,20260527,0.641,0.644,0.628,0.630,3424402.00,218182.120
